In [1]:
import sys
sys.path.append("E:\데이터분석\workspace")
from helpers import *

import numpy as np
from IPython.display import display, Markdown
import pandas as pd

<>:2: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<>:2: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
C:\Users\user\AppData\Local\Temp\ipykernel_6012\1299606760.py:2: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
  sys.path.append("E:\데이터분석\workspace")


## 데이터로드

In [2]:
# 1. 데이터 로드
# 0 결측치 제거, 파생변수 미포함, 별점 컬럼o
df = pd.read_excel("df_cleaned.xlsx")
df.head()

,국가,제품_순위,Y,보습,향,발림성,자극,용기,배송,기능,가격,브랜드충성도
0,러시아,1,1,0,0,1,0,1,0,1,0,0
1,러시아,1,1,1,0,0,0,0,0,0,0,0
2,러시아,1,1,1,1,0,0,1,0,1,0,0
3,러시아,1,1,0,1,1,0,1,1,0,0,0
4,러시아,1,1,1,0,0,0,0,0,0,0,0


## 변수 분포 확인

In [3]:
attrs = ['보습', '향', '발림성', '자극', '용기', '배송', '기능', '가격', '브랜드충성도']
countries_all = ['한국', '러시아', '미국', '일본']
countries_nonref = ['러시아', '미국', '일본']   # 한국 = 기준범주(reference)

country_dummy_cols = [f'국가_{c}' for c in countries_nonref]
interaction_cols = [f'국가_{c}_x_{a}' for c in countries_nonref for a in attrs]

print('=== Y 분포 ===')
print(df['Y'].value_counts().sort_index())

print('\n=== 국가별 표본 수 ===')
print(df['국가'].value_counts())

print('\n=== 속성 언급 비율 ===')
print(df[attrs].mean().round(3).sort_values(ascending=False))

print('\n결측치 개수:', df.isna().sum().sum())
print('속성값이 모두 0인 행(사전 제거 확인):', (df[attrs].sum(axis=1) == 0).sum())

=== Y 분포 ===
Y
0    2336
1    4420
Name: count, dtype: int64

=== 국가별 표본 수 ===
국가
한국     1832
미국     1818
러시아    1608
일본     1498
Name: count, dtype: int64

=== 속성 언급 비율 ===
발림성       0.467
보습        0.459
자극        0.451
기능        0.366
용기        0.206
향         0.163
브랜드충성도    0.107
가격        0.081
배송        0.063
dtype: float64

결측치 개수: 0
속성값이 모두 0인 행(사전 제거 확인): 0


In [4]:
# 국가별 x 속성별 언급 교차표 (셀별 표본수 확인 -> 상호작용 추정 가능 여부 점검)
cross_counts = pd.concat(
    {a: df.groupby('국가', observed=True)[a].sum() for a in attrs}, axis=1
)
cross_counts

,보습,향,발림성,자극,용기,배송,기능,가격,브랜드충성도
국가,,,,,,,,,
러시아,385,630,756,239,562,174,768,76,125
미국,1037,269,843,1010,269,62,691,130,152
일본,661,155,633,651,331,98,665,190,316
한국,1016,45,921,1144,230,93,348,152,130


## 국가 더미화, 국가x속성 파생변수 추가

In [5]:
# 2. 속성 리스트 (가설 1~4에 맞게 조정 가능, 현재는 전체 9개)
attrs = ['보습', '향', '발림성', '자극', '용기', '배송', '기능', '가격', '브랜드충성도']

# 3. 국가 더미화 (기준 범주 = 한국)
country_order = ['한국', '러시아', '미국', '일본']  # 한국을 맨 앞에 둬야 drop_first 시 한국이 기준으로 빠짐
country_cat = pd.Categorical(df['국가'], categories=country_order)
country_dummies = pd.get_dummies(country_cat, prefix='국가', drop_first=True).astype(int)
# -> 국가_러시아, 국가_미국, 국가_일본 생성 (한국은 기준이라 컬럼 없음)

# 4. 국가더미 x 속성더미 상호작용항 생성
interaction_df = pd.DataFrame({
    f'{c}_x_{a}': country_dummies[c] * df[a]
    for c in country_dummies.columns
    for a in attrs
})

# 5. 원본 데이터셋에 파생변수(국가 더미 + 상호작용항) 결합
df1 = pd.concat([df, country_dummies, interaction_df], axis=1)
display(df1)
df1.info()

,국가,제품_순위,Y,보습,향,발림성,자극,용기,배송,기능,...,국가_미국_x_브랜드충성도,국가_일본_x_보습,국가_일본_x_향,국가_일본_x_발림성,국가_일본_x_자극,국가_일본_x_용기,국가_일본_x_배송,국가_일본_x_기능,국가_일본_x_가격,국가_일본_x_브랜드충성도
0,러시아,1,1,0,0,1,0,1,0,1,...,0,0,0,0,0,0,0,0,0,0
1,러시아,1,1,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,러시아,1,1,1,1,0,0,1,0,1,...,0,0,0,0,0,0,0,0,0,0
3,러시아,1,1,0,1,1,0,1,1,0,...,0,0,0,0,0,0,0,0,0,0
4,러시아,1,1,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6751,한국,10,0,1,1,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6752,한국,10,0,0,0,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6753,한국,10,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6754,한국,10,0,1,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0


<class 'pandas.DataFrame'>
RangeIndex: 6756 entries, 0 to 6755
Data columns (total 42 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   국가               6756 non-null   str  
 1   제품_순위            6756 non-null   int64
 2   Y                6756 non-null   int64
 3   보습               6756 non-null   int64
 4   향                6756 non-null   int64
 5   발림성              6756 non-null   int64
 6   자극               6756 non-null   int64
 7   용기               6756 non-null   int64
 8   배송               6756 non-null   int64
 9   기능               6756 non-null   int64
 10  가격               6756 non-null   int64
 11  브랜드충성도           6756 non-null   int64
 12  국가_러시아           6756 non-null   int64
 13  국가_미국            6756 non-null   int64
 14  국가_일본            6756 non-null   int64
 15  국가_러시아_x_보습      6756 non-null   int64
 16  국가_러시아_x_향       6756 non-null   int64
 17  국가_러시아_x_발림성     6756 non-null   int64
 18  국가_러시아_x_자극      67

## 컬럼 제거

In [6]:
#df_final = df1.drop(columns=['국가', '별점'])

## 자료형 변환

In [7]:
df_final=my_qtcheck.set_type(df1, as_category=['Y'])

<class 'pandas.DataFrame'>
RangeIndex: 6756 entries, 0 to 6755
Data columns (total 42 columns):
 #   Column           Non-Null Count  Dtype   
---  ------           --------------  -----   
 0   국가               6756 non-null   str     
 1   제품_순위            6756 non-null   int64   
 2   Y                6756 non-null   category
 3   보습               6756 non-null   int64   
 4   향                6756 non-null   int64   
 5   발림성              6756 non-null   int64   
 6   자극               6756 non-null   int64   
 7   용기               6756 non-null   int64   
 8   배송               6756 non-null   int64   
 9   기능               6756 non-null   int64   
 10  가격               6756 non-null   int64   
 11  브랜드충성도           6756 non-null   int64   
 12  국가_러시아           6756 non-null   int64   
 13  국가_미국            6756 non-null   int64   
 14  국가_일본            6756 non-null   int64   
 15  국가_러시아_x_보습      6756 non-null   int64   
 16  국가_러시아_x_향       6756 non-null   int64   
 17  국가_러시아

## 로지스틱회귀 - 국가x속성 제거, full 비교

In [21]:
X_reduced = df_final[country_dummy_cols + attrs + ['Y']]
X_full = df_final[country_dummy_cols + attrs + interaction_cols + ['Y']]

print('모형 A 설계행렬 shape:', X_reduced.shape)
print('모형 B 설계행렬 shape:', X_full.shape)

모형 A 설계행렬 shape: (6756, 13)
모형 B 설계행렬 shape: (6756, 40)


## 로지스틱회귀 - 모형A (국가x속성 제거)

In [9]:
fit1 = my_logit.fit_model(X_reduced, y='Y')
my_logit.report_variables(fit1, X_reduced)

,종속변수,독립변수,B,βstd,표준오차,z,유의확률,B 95% 하한,B 95% 상한,오즈비(OR),OR 95% 하한,OR 95% 상한,공차,VIF
0,Y,국가_러시아,1.408073,0.599694,0.100454,14.017037,0.0000,1.211186,1.604960,4.088070,3.357464,4.977660,0.493754,2.025298
1,Y,국가_일본,1.228239,0.510260,0.087708,14.003791,0.0000,1.056335,1.400143,3.415209,2.875812,4.055778,0.639352,1.564084
2,Y,배송,-1.053808,-0.256440,0.117197,-8.991730,0.0000,-1.283511,-0.824105,0.348608,0.277063,0.438627,0.936110,1.068250
3,Y,자극,-0.909223,-0.452417,0.061425,-14.802257,0.0000,-1.029613,-0.788833,0.402837,0.357145,0.454375,0.832971,1.200522
4,Y,보습,0.855699,0.426419,0.061889,13.826361,0.0000,0.734399,0.976999,2.353018,2.084228,2.656471,0.890186,1.123361
5,Y,기능,0.670133,0.322814,0.064329,10.417245,0.0000,0.544050,0.796216,1.954497,1.722971,2.217135,0.921926,1.084686
6,Y,브랜드충성도,0.514331,0.159009,0.101069,5.088918,0.0000,0.316240,0.712423,1.672520,1.371959,2.038925,0.950854,1.051687
7,Y,가격,0.396964,0.108383,0.111705,3.553679,0.0004,0.178026,0.615901,1.487302,1.194856,1.851324,0.958618,1.043168
8,Y,향,0.297875,0.109943,0.092197,3.230840,0.0012,0.117172,0.478579,1.346994,1.124312,1.613779,0.834826,1.197854
9,Y,발림성,0.252253,0.125856,0.059415,4.245590,0.0000,0.135801,0.368705,1.286922,1.145454,1.445861,0.939153,1.064789


## 로지스틱회귀 - 모형 B (full)

In [10]:
fit2 = my_logit.fit_model(X_full, y='Y')
my_logit.report_variables(fit2, X_full)

,종속변수,독립변수,B,βstd,표준오차,z,유의확률,B 95% 하한,B 95% 상한,오즈비(OR),OR 95% 하한,OR 95% 상한,공차,VIF
0,Y,배송,-1.264850,-0.307796,0.250541,-5.048473,0.0000,-1.755901,-0.773798,0.282282,0.172751,0.461258,0.214204,4.668444
1,Y,국가_일본,1.197615,0.497538,0.217597,5.503817,0.0000,0.771132,1.624098,3.312208,2.162213,5.073839,0.110621,9.039909
2,Y,국가_미국_x_보습,1.162461,0.419054,0.149163,7.793203,0.0000,0.870106,1.454815,3.197792,2.387163,4.283692,0.238927,4.185379
3,Y,국가_일본_x_브랜드충성도,1.095359,0.231305,0.282214,3.881308,0.0001,0.542230,1.648489,2.990257,1.719838,5.199116,0.264973,3.773970
4,Y,자극,-0.918744,-0.457155,0.105342,-8.721548,0.0000,-1.125211,-0.712278,0.399020,0.324584,0.490525,0.242207,4.128697
5,Y,국가_러시아_x_브랜드충성도,0.913793,0.123150,0.351489,2.599777,0.0093,0.224887,1.602698,2.493763,1.252181,4.966416,0.469831,2.128427
6,Y,국가_미국,-0.830576,-0.368378,0.186533,-4.452701,0.0000,-1.196173,-0.464978,0.435798,0.302349,0.628149,0.102029,9.801145
7,Y,국가_러시아_x_향,0.790638,0.229921,0.360992,2.190183,0.0285,0.083107,1.498169,2.204802,1.086658,4.473489,0.067517,14.810985
8,Y,국가_미국_x_브랜드충성도,0.757569,0.112354,0.272405,2.781038,0.0054,0.223665,1.291473,2.133084,1.250652,3.638142,0.429226,2.329776
9,Y,기능,0.754335,0.363375,0.129036,5.845940,0.0000,0.501429,1.007240,2.126196,1.651080,2.738033,0.176574,5.663334


## 국가별 실제 효과 — Odds Ratio
상호작용 계수는 **한국과의 차이**를 보여준다.  
따라서 각 국가에서 해당 속성이 만족도와 실제로 어떤 관계를 가지는지 확인하기 위해  
**주효과 + 해당 국가의 상호작용 효과**를 합산한 뒤 Odds Ratio(OR)로 변환하였다.

In [11]:
import numpy as np
import pandas as pd

rows = []
for attr in attrs:
    for country in countries_all:
        if country == '한국':
            combo = attr  # 기준국가는 상호작용항 없이 속성 계수 그대로
        else:
            combo = f"{attr} + 국가_{country}_x_{attr}"

        tt = fit2.t_test(combo)
        coef   = tt.effect[0]
        pval   = np.atleast_1d(tt.pvalue)[0]
        ci_low, ci_high = tt.conf_int()[0]

        rows.append({
            '속성': attr,
            '국가': country,
            'OR': np.exp(coef),
            'OR_CI_하한': np.exp(ci_low),
            'OR_CI_상한': np.exp(ci_high),
            'p값': pval,
            '유의': '*' if pval < .05 else ' n.s.'
        })

or_ci_table = pd.DataFrame(rows)

# 15절 표 형태로 피벗 (속성 x 국가, "OR* (하한-상한)" 문자열)
def fmt(r):
    star = r['유의']
    return f"{r['OR']:.2f}{star} ({r['OR_CI_하한']:.2f}–{r['OR_CI_상한']:.2f})"   

or_ci_table['표시값'] = or_ci_table.apply(fmt, axis=1)
table15 = or_ci_table.pivot(index='속성', columns='국가', values='표시값')
table15 = table15[countries_all]  # 한국·러시아·미국·일본 순서 정리
table15

국가,한국,러시아,미국,일본
속성,,,,
가격,1.26 n.s. (0.88–1.81),1.76 n.s. (0.84–3.71),1.33 n.s. (0.87–2.01),1.83* (1.13–2.96)
기능,2.13* (1.65–2.74),1.34 n.s. (0.99–1.82),2.85* (2.28–3.55),1.15 n.s. (0.87–1.52)
발림성,1.28* (1.05–1.55),1.97* (1.44–2.70),1.05 n.s. (0.85–1.31),1.30 n.s. (0.98–1.73)
배송,0.28* (0.17–0.46),0.45* (0.29–0.68),0.38* (0.20–0.72),0.21* (0.13–0.34)
보습,1.32* (1.09–1.62),2.10* (1.46–3.04),4.23* (3.42–5.25),2.53* (1.88–3.42)
브랜드충성도,0.86 n.s. (0.59–1.26),2.15* (1.21–3.84),1.84* (1.26–2.69),2.58* (1.72–3.88)
용기,0.97 n.s. (0.72–1.30),1.31 n.s. (0.94–1.82),0.77 n.s. (0.57–1.04),0.98 n.s. (0.70–1.38)
자극,0.40* (0.32–0.49),0.38* (0.27–0.54),0.45* (0.36–0.55),0.28* (0.21–0.38)
향,1.05 n.s. (0.57–1.96),2.32* (1.65–3.26),0.91 n.s. (0.68–1.22),1.32 n.s. (0.83–2.11)


## 모형적합도 보고
### 모형A

In [12]:
Markdown(my_logit.report_fitness(fit1))

**Note. n = 6756. LL = -3607.562, LL-Null = -4356.178, LLR χ²(12) = 1497.234, p < 0.001, Pseudo R² = 0.172**

Y를 종속변수로, 국가_러시아, 국가_미국, 국가_일본, 보습, 향, 발림성, 자극, 용기, 배송, 기능, 가격, 브랜드충성도(을)를 독립변수로 한 이항 로지스틱 회귀분석 결과, 모형은 통계적으로 유의하였다.

> LLR χ²(12) = 1497.234, p < 0.001, Pseudo R² = 0.172.

즉, 모형의 유사결정계수는 0.172로 양호한 적합 수준을 보였다.

> ※ Pseudo R²는 선형회귀의 R²처럼 '분산 설명 비율'로 해석하지 않는다. 일반적으로 **0.2~0.4** 구간이면 매우 우수한 적합으로 본다.

### 모형B

In [13]:
Markdown(my_logit.report_fitness(fit2))

**Note. n = 6756. LL = -3519.986, LL-Null = -4356.178, LLR χ²(39) = 1672.386, p < 0.001, Pseudo R² = 0.192**

Y를 종속변수로, 국가_러시아, 국가_미국, 국가_일본, 보습, 향, 발림성, 자극, 용기, 배송, 기능, 가격, 브랜드충성도, 국가_러시아_x_보습, 국가_러시아_x_향, 국가_러시아_x_발림성, 국가_러시아_x_자극, 국가_러시아_x_용기, 국가_러시아_x_배송, 국가_러시아_x_기능, 국가_러시아_x_가격, 국가_러시아_x_브랜드충성도, 국가_미국_x_보습, 국가_미국_x_향, 국가_미국_x_발림성, 국가_미국_x_자극, 국가_미국_x_용기, 국가_미국_x_배송, 국가_미국_x_기능, 국가_미국_x_가격, 국가_미국_x_브랜드충성도, 국가_일본_x_보습, 국가_일본_x_향, 국가_일본_x_발림성, 국가_일본_x_자극, 국가_일본_x_용기, 국가_일본_x_배송, 국가_일본_x_기능, 국가_일본_x_가격, 국가_일본_x_브랜드충성도(을)를 독립변수로 한 이항 로지스틱 회귀분석 결과, 모형은 통계적으로 유의하였다.

> LLR χ²(39) = 1672.386, p < 0.001, Pseudo R² = 0.192.

즉, 모형의 유사결정계수는 0.192로 양호한 적합 수준을 보였다.

> ※ Pseudo R²는 선형회귀의 R²처럼 '분산 설명 비율'로 해석하지 않는다. 일반적으로 **0.2~0.4** 구간이면 매우 우수한 적합으로 본다.

> 국가 x 속성항이 조금 더 유의한 결과값을 보임 (Pseudo R²)

## 모형 A vs 모형 B 우도비 검정

In [24]:
from scipy import stats

lr_stat = -2 * (fit1.llf - fit2.llf)       # 모형 A vs 모형 B 우도비 χ²
df_diff = fit2.df_model - fit1.df_model     # 추가된 변수 수 (상호작용항 27개)
p_value = stats.chi2.sf(lr_stat, df_diff)

print(f"LR χ²({df_diff:.0f}) = {lr_stat:.3f}, p = {p_value:.4f}")

LR χ²(27) = 175.152, p = 0.0000


## 독립변수 보고

### 모형A

In [16]:
display(my_logit.report_variables(fit1, X_reduced))
Markdown(my_logit.report_variables_text(fit1))

,종속변수,독립변수,B,βstd,표준오차,z,유의확률,B 95% 하한,B 95% 상한,오즈비(OR),OR 95% 하한,OR 95% 상한,공차,VIF
0,Y,국가_러시아,1.408073,0.599694,0.100454,14.017037,0.0000,1.211186,1.604960,4.088070,3.357464,4.977660,0.493754,2.025298
1,Y,국가_일본,1.228239,0.510260,0.087708,14.003791,0.0000,1.056335,1.400143,3.415209,2.875812,4.055778,0.639352,1.564084
2,Y,배송,-1.053808,-0.256440,0.117197,-8.991730,0.0000,-1.283511,-0.824105,0.348608,0.277063,0.438627,0.936110,1.068250
3,Y,자극,-0.909223,-0.452417,0.061425,-14.802257,0.0000,-1.029613,-0.788833,0.402837,0.357145,0.454375,0.832971,1.200522
4,Y,보습,0.855699,0.426419,0.061889,13.826361,0.0000,0.734399,0.976999,2.353018,2.084228,2.656471,0.890186,1.123361
5,Y,기능,0.670133,0.322814,0.064329,10.417245,0.0000,0.544050,0.796216,1.954497,1.722971,2.217135,0.921926,1.084686
6,Y,브랜드충성도,0.514331,0.159009,0.101069,5.088918,0.0000,0.316240,0.712423,1.672520,1.371959,2.038925,0.950854,1.051687
7,Y,가격,0.396964,0.108383,0.111705,3.553679,0.0004,0.178026,0.615901,1.487302,1.194856,1.851324,0.958618,1.043168
8,Y,향,0.297875,0.109943,0.092197,3.230840,0.0012,0.117172,0.478579,1.346994,1.124312,1.613779,0.834826,1.197854
9,Y,발림성,0.252253,0.125856,0.059415,4.245590,0.0000,0.135801,0.368705,1.286922,1.145454,1.445861,0.939153,1.064789


- **국가_러시아**의 회귀계수는 **B = 1.4081**, 오즈비는 **OR = 4.0881**로 나타났으며, 이는 **Y**에 유의한 요인임을 의미한다. (**z = 14.02**, **p < .001**) 즉, **국가_러시아**가 1 증가할 때 Y가 1(사건 발생)이 될 오즈는 평균적으로 약 **308.8% 증가**하는 것으로 해석된다.
- **국가_미국**의 회귀계수는 **B = -0.0976**, 오즈비는 **OR = 0.907**로 나타났으며, 이는 **Y**에 유의하지 않은 요인임을 의미한다. (**z = -1.33**, **p = .183**) 즉, **국가_미국**가 1 증가할 때 Y가 1(사건 발생)이 될 오즈는 평균적으로 약 **9.3% 감소**하는 것으로 해석된다.
- **국가_일본**의 회귀계수는 **B = 1.2282**, 오즈비는 **OR = 3.4152**로 나타났으며, 이는 **Y**에 유의한 요인임을 의미한다. (**z = 14.0**, **p < .001**) 즉, **국가_일본**가 1 증가할 때 Y가 1(사건 발생)이 될 오즈는 평균적으로 약 **241.5% 증가**하는 것으로 해석된다.
- **보습**의 회귀계수는 **B = 0.8557**, 오즈비는 **OR = 2.353**로 나타났으며, 이는 **Y**에 유의한 요인임을 의미한다. (**z = 13.83**, **p < .001**) 즉, **보습**가 1 증가할 때 Y가 1(사건 발생)이 될 오즈는 평균적으로 약 **135.3% 증가**하는 것으로 해석된다.
- **향**의 회귀계수는 **B = 0.2979**, 오즈비는 **OR = 1.347**로 나타났으며, 이는 **Y**에 유의한 요인임을 의미한다. (**z = 3.23**, **p = .001**) 즉, **향**가 1 증가할 때 Y가 1(사건 발생)이 될 오즈는 평균적으로 약 **34.7% 증가**하는 것으로 해석된다.
- **발림성**의 회귀계수는 **B = 0.2523**, 오즈비는 **OR = 1.2869**로 나타났으며, 이는 **Y**에 유의한 요인임을 의미한다. (**z = 4.25**, **p < .001**) 즉, **발림성**가 1 증가할 때 Y가 1(사건 발생)이 될 오즈는 평균적으로 약 **28.7% 증가**하는 것으로 해석된다.
- **자극**의 회귀계수는 **B = -0.9092**, 오즈비는 **OR = 0.4028**로 나타났으며, 이는 **Y**에 유의한 요인임을 의미한다. (**z = -14.8**, **p < .001**) 즉, **자극**가 1 증가할 때 Y가 1(사건 발생)이 될 오즈는 평균적으로 약 **59.7% 감소**하는 것으로 해석된다.
- **용기**의 회귀계수는 **B = 0.0026**, 오즈비는 **OR = 1.0026**로 나타났으며, 이는 **Y**에 유의하지 않은 요인임을 의미한다. (**z = 0.03**, **p = .973**) 즉, **용기**가 1 증가할 때 Y가 1(사건 발생)이 될 오즈는 평균적으로 약 **0.3% 증가**하는 것으로 해석된다.
- **배송**의 회귀계수는 **B = -1.0538**, 오즈비는 **OR = 0.3486**로 나타났으며, 이는 **Y**에 유의한 요인임을 의미한다. (**z = -8.99**, **p < .001**) 즉, **배송**가 1 증가할 때 Y가 1(사건 발생)이 될 오즈는 평균적으로 약 **65.1% 감소**하는 것으로 해석된다.
- **기능**의 회귀계수는 **B = 0.6701**, 오즈비는 **OR = 1.9545**로 나타났으며, 이는 **Y**에 유의한 요인임을 의미한다. (**z = 10.42**, **p < .001**) 즉, **기능**가 1 증가할 때 Y가 1(사건 발생)이 될 오즈는 평균적으로 약 **95.4% 증가**하는 것으로 해석된다.
- **가격**의 회귀계수는 **B = 0.397**, 오즈비는 **OR = 1.4873**로 나타났으며, 이는 **Y**에 유의한 요인임을 의미한다. (**z = 3.55**, **p < .001**) 즉, **가격**가 1 증가할 때 Y가 1(사건 발생)이 될 오즈는 평균적으로 약 **48.7% 증가**하는 것으로 해석된다.
- **브랜드충성도**의 회귀계수는 **B = 0.5143**, 오즈비는 **OR = 1.6725**로 나타났으며, 이는 **Y**에 유의한 요인임을 의미한다. (**z = 5.09**, **p < .001**) 즉, **브랜드충성도**가 1 증가할 때 Y가 1(사건 발생)이 될 오즈는 평균적으로 약 **67.3% 증가**하는 것으로 해석된다.

> ※ 오즈비(OR)가 1보다 크면 사건 발생 오즈가 증가, 1보다 작으면 감소함을 뜻한다. 유의확률이 유의수준보다 큰(=유의하지 않은) 변수는 효과가 통계적으로 확인되지 않았으므로 오즈비 해석에 주의한다. 더미변수의 오즈비는 '기준(drop_first로 제외된) 범주' 대비 값이다.

### 모형B

In [17]:
display(my_logit.report_variables(fit2, X_full))
Markdown(my_logit.report_variables_text(fit2))

,종속변수,독립변수,B,βstd,표준오차,z,유의확률,B 95% 하한,B 95% 상한,오즈비(OR),OR 95% 하한,OR 95% 상한,공차,VIF
0,Y,배송,-1.264850,-0.307796,0.250541,-5.048473,0.0000,-1.755901,-0.773798,0.282282,0.172751,0.461258,0.214204,4.668444
1,Y,국가_일본,1.197615,0.497538,0.217597,5.503817,0.0000,0.771132,1.624098,3.312208,2.162213,5.073839,0.110621,9.039909
2,Y,국가_미국_x_보습,1.162461,0.419054,0.149163,7.793203,0.0000,0.870106,1.454815,3.197792,2.387163,4.283692,0.238927,4.185379
3,Y,국가_일본_x_브랜드충성도,1.095359,0.231305,0.282214,3.881308,0.0001,0.542230,1.648489,2.990257,1.719838,5.199116,0.264973,3.773970
4,Y,자극,-0.918744,-0.457155,0.105342,-8.721548,0.0000,-1.125211,-0.712278,0.399020,0.324584,0.490525,0.242207,4.128697
5,Y,국가_러시아_x_브랜드충성도,0.913793,0.123150,0.351489,2.599777,0.0093,0.224887,1.602698,2.493763,1.252181,4.966416,0.469831,2.128427
6,Y,국가_미국,-0.830576,-0.368378,0.186533,-4.452701,0.0000,-1.196173,-0.464978,0.435798,0.302349,0.628149,0.102029,9.801145
7,Y,국가_러시아_x_향,0.790638,0.229921,0.360992,2.190183,0.0285,0.083107,1.498169,2.204802,1.086658,4.473489,0.067517,14.810985
8,Y,국가_미국_x_브랜드충성도,0.757569,0.112354,0.272405,2.781038,0.0054,0.223665,1.291473,2.133084,1.250652,3.638142,0.429226,2.329776
9,Y,기능,0.754335,0.363375,0.129036,5.845940,0.0000,0.501429,1.007240,2.126196,1.651080,2.738033,0.176574,5.663334


- **국가_러시아**의 회귀계수는 **B = 0.7182**, 오즈비는 **OR = 2.0508**로 나타났으며, 이는 **Y**에 유의한 요인임을 의미한다. (**z = 3.6**, **p < .001**) 즉, **국가_러시아**가 1 증가할 때 Y가 1(사건 발생)이 될 오즈는 평균적으로 약 **105.1% 증가**하는 것으로 해석된다.
- **국가_미국**의 회귀계수는 **B = -0.8306**, 오즈비는 **OR = 0.4358**로 나타났으며, 이는 **Y**에 유의한 요인임을 의미한다. (**z = -4.45**, **p < .001**) 즉, **국가_미국**가 1 증가할 때 Y가 1(사건 발생)이 될 오즈는 평균적으로 약 **56.4% 감소**하는 것으로 해석된다.
- **국가_일본**의 회귀계수는 **B = 1.1976**, 오즈비는 **OR = 3.3122**로 나타났으며, 이는 **Y**에 유의한 요인임을 의미한다. (**z = 5.5**, **p < .001**) 즉, **국가_일본**가 1 증가할 때 Y가 1(사건 발생)이 될 오즈는 평균적으로 약 **231.2% 증가**하는 것으로 해석된다.
- **보습**의 회귀계수는 **B = 0.2806**, 오즈비는 **OR = 1.3239**로 나타났으며, 이는 **Y**에 유의한 요인임을 의미한다. (**z = 2.76**, **p = .006**) 즉, **보습**가 1 증가할 때 Y가 1(사건 발생)이 될 오즈는 평균적으로 약 **32.4% 증가**하는 것으로 해석된다.
- **향**의 회귀계수는 **B = 0.0513**, 오즈비는 **OR = 1.0527**로 나타났으며, 이는 **Y**에 유의하지 않은 요인임을 의미한다. (**z = 0.16**, **p = .871**) 즉, **향**가 1 증가할 때 Y가 1(사건 발생)이 될 오즈는 평균적으로 약 **5.3% 증가**하는 것으로 해석된다.
- **발림성**의 회귀계수는 **B = 0.2446**, 오즈비는 **OR = 1.2771**로 나타났으며, 이는 **Y**에 유의한 요인임을 의미한다. (**z = 2.46**, **p = .014**) 즉, **발림성**가 1 증가할 때 Y가 1(사건 발생)이 될 오즈는 평균적으로 약 **27.7% 증가**하는 것으로 해석된다.
- **자극**의 회귀계수는 **B = -0.9187**, 오즈비는 **OR = 0.399**로 나타났으며, 이는 **Y**에 유의한 요인임을 의미한다. (**z = -8.72**, **p < .001**) 즉, **자극**가 1 증가할 때 Y가 1(사건 발생)이 될 오즈는 평균적으로 약 **60.1% 감소**하는 것으로 해석된다.
- **용기**의 회귀계수는 **B = -0.0297**, 오즈비는 **OR = 0.9708**로 나타났으며, 이는 **Y**에 유의하지 않은 요인임을 의미한다. (**z = -0.2**, **p = .843**) 즉, **용기**가 1 증가할 때 Y가 1(사건 발생)이 될 오즈는 평균적으로 약 **2.9% 감소**하는 것으로 해석된다.
- **배송**의 회귀계수는 **B = -1.2648**, 오즈비는 **OR = 0.2823**로 나타났으며, 이는 **Y**에 유의한 요인임을 의미한다. (**z = -5.05**, **p < .001**) 즉, **배송**가 1 증가할 때 Y가 1(사건 발생)이 될 오즈는 평균적으로 약 **71.8% 감소**하는 것으로 해석된다.
- **기능**의 회귀계수는 **B = 0.7543**, 오즈비는 **OR = 2.1262**로 나타났으며, 이는 **Y**에 유의한 요인임을 의미한다. (**z = 5.85**, **p < .001**) 즉, **기능**가 1 증가할 때 Y가 1(사건 발생)이 될 오즈는 평균적으로 약 **112.6% 증가**하는 것으로 해석된다.
- **가격**의 회귀계수는 **B = 0.2303**, 오즈비는 **OR = 1.259**로 나타났으며, 이는 **Y**에 유의하지 않은 요인임을 의미한다. (**z = 1.24**, **p = .213**) 즉, **가격**가 1 증가할 때 Y가 1(사건 발생)이 될 오즈는 평균적으로 약 **25.9% 증가**하는 것으로 해석된다.
- **브랜드충성도**의 회귀계수는 **B = -0.1471**, 오즈비는 **OR = 0.8632**로 나타났으며, 이는 **Y**에 유의하지 않은 요인임을 의미한다. (**z = -0.77**, **p = .441**) 즉, **브랜드충성도**가 1 증가할 때 Y가 1(사건 발생)이 될 오즈는 평균적으로 약 **13.7% 감소**하는 것으로 해석된다.
- **국가_러시아_x_보습**의 회귀계수는 **B = 0.4632**, 오즈비는 **OR = 1.5892**로 나타났으며, 이는 **Y**에 유의한 요인임을 의미한다. (**z = 2.17**, **p = .030**) 즉, **국가_러시아_x_보습**가 1 증가할 때 Y가 1(사건 발생)이 될 오즈는 평균적으로 약 **58.9% 증가**하는 것으로 해석된다.
- **국가_러시아_x_향**의 회귀계수는 **B = 0.7906**, 오즈비는 **OR = 2.2048**로 나타났으며, 이는 **Y**에 유의한 요인임을 의미한다. (**z = 2.19**, **p = .029**) 즉, **국가_러시아_x_향**가 1 증가할 때 Y가 1(사건 발생)이 될 오즈는 평균적으로 약 **120.5% 증가**하는 것으로 해석된다.
- **국가_러시아_x_발림성**의 회귀계수는 **B = 0.4353**, 오즈비는 **OR = 1.5455**로 나타났으며, 이는 **Y**에 유의한 요인임을 의미한다. (**z = 2.31**, **p = .021**) 즉, **국가_러시아_x_발림성**가 1 증가할 때 Y가 1(사건 발생)이 될 오즈는 평균적으로 약 **54.5% 증가**하는 것으로 해석된다.
- **국가_러시아_x_자극**의 회귀계수는 **B = -0.0384**, 오즈비는 **OR = 0.9623**로 나타났으며, 이는 **Y**에 유의하지 않은 요인임을 의미한다. (**z = -0.19**, **p = .851**) 즉, **국가_러시아_x_자극**가 1 증가할 때 Y가 1(사건 발생)이 될 오즈는 평균적으로 약 **3.8% 감소**하는 것으로 해석된다.
- **국가_러시아_x_용기**의 회귀계수는 **B = 0.2985**, 오즈비는 **OR = 1.3478**로 나타났으며, 이는 **Y**에 유의하지 않은 요인임을 의미한다. (**z = 1.33**, **p = .185**) 즉, **국가_러시아_x_용기**가 1 증가할 때 Y가 1(사건 발생)이 될 오즈는 평균적으로 약 **34.8% 증가**하는 것으로 해석된다.
- **국가_러시아_x_배송**의 회귀계수는 **B = 0.4585**, 오즈비는 **OR = 1.5818**로 나타났으며, 이는 **Y**에 유의하지 않은 요인임을 의미한다. (**z = 1.39**, **p = .166**) 즉, **국가_러시아_x_배송**가 1 증가할 때 Y가 1(사건 발생)이 될 오즈는 평균적으로 약 **58.2% 증가**하는 것으로 해석된다.
- **국가_러시아_x_기능**의 회귀계수는 **B = -0.4613**, 오즈비는 **OR = 0.6305**로 나타났으며, 이는 **Y**에 유의한 요인임을 의미한다. (**z = -2.29**, **p = .022**) 즉, **국가_러시아_x_기능**가 1 증가할 때 Y가 1(사건 발생)이 될 오즈는 평균적으로 약 **37.0% 감소**하는 것으로 해석된다.
- **국가_러시아_x_가격**의 회귀계수는 **B = 0.3362**, 오즈비는 **OR = 1.3997**로 나타났으며, 이는 **Y**에 유의하지 않은 요인임을 의미한다. (**z = 0.8**, **p = .427**) 즉, **국가_러시아_x_가격**가 1 증가할 때 Y가 1(사건 발생)이 될 오즈는 평균적으로 약 **40.0% 증가**하는 것으로 해석된다.
- **국가_러시아_x_브랜드충성도**의 회귀계수는 **B = 0.9138**, 오즈비는 **OR = 2.4938**로 나타났으며, 이는 **Y**에 유의한 요인임을 의미한다. (**z = 2.6**, **p = .009**) 즉, **국가_러시아_x_브랜드충성도**가 1 증가할 때 Y가 1(사건 발생)이 될 오즈는 평균적으로 약 **149.4% 증가**하는 것으로 해석된다.
- **국가_미국_x_보습**의 회귀계수는 **B = 1.1625**, 오즈비는 **OR = 3.1978**로 나타났으며, 이는 **Y**에 유의한 요인임을 의미한다. (**z = 7.79**, **p < .001**) 즉, **국가_미국_x_보습**가 1 증가할 때 Y가 1(사건 발생)이 될 오즈는 평균적으로 약 **219.8% 증가**하는 것으로 해석된다.
- **국가_미국_x_향**의 회귀계수는 **B = -0.1432**, 오즈비는 **OR = 0.8666**로 나타났으며, 이는 **Y**에 유의하지 않은 요인임을 의미한다. (**z = -0.41**, **p = .682**) 즉, **국가_미국_x_향**가 1 증가할 때 Y가 1(사건 발생)이 될 오즈는 평균적으로 약 **13.3% 감소**하는 것으로 해석된다.
- **국가_미국_x_발림성**의 회귀계수는 **B = -0.1941**, 오즈비는 **OR = 0.8236**로 나타났으며, 이는 **Y**에 유의하지 않은 요인임을 의미한다. (**z = -1.31**, **p = .192**) 즉, **국가_미국_x_발림성**가 1 증가할 때 Y가 1(사건 발생)이 될 오즈는 평균적으로 약 **17.6% 감소**하는 것으로 해석된다.
- **국가_미국_x_자극**의 회귀계수는 **B = 0.1115**, 오즈비는 **OR = 1.118**로 나타났으며, 이는 **Y**에 유의하지 않은 요인임을 의미한다. (**z = 0.73**, **p = .467**) 즉, **국가_미국_x_자극**가 1 증가할 때 Y가 1(사건 발생)이 될 오즈는 평균적으로 약 **11.8% 증가**하는 것으로 해석된다.
- **국가_미국_x_용기**의 회귀계수는 **B = -0.2292**, 오즈비는 **OR = 0.7952**로 나타났으며, 이는 **Y**에 유의하지 않은 요인임을 의미한다. (**z = -1.07**, **p = .286**) 즉, **국가_미국_x_용기**가 1 증가할 때 Y가 1(사건 발생)이 될 오즈는 평균적으로 약 **20.5% 감소**하는 것으로 해석된다.
- **국가_미국_x_배송**의 회귀계수는 **B = 0.2963**, 오즈비는 **OR = 1.3449**로 나타났으며, 이는 **Y**에 유의하지 않은 요인임을 의미한다. (**z = 0.72**, **p = .469**) 즉, **국가_미국_x_배송**가 1 증가할 때 Y가 1(사건 발생)이 될 오즈는 평균적으로 약 **34.5% 증가**하는 것으로 해석된다.
- **국가_미국_x_기능**의 회귀계수는 **B = 0.2918**, 오즈비는 **OR = 1.3388**로 나타났으며, 이는 **Y**에 유의하지 않은 요인임을 의미한다. (**z = 1.7**, **p = .088**) 즉, **국가_미국_x_기능**가 1 증가할 때 Y가 1(사건 발생)이 될 오즈는 평균적으로 약 **33.9% 증가**하는 것으로 해석된다.
- **국가_미국_x_가격**의 회귀계수는 **B = 0.0524**, 오즈비는 **OR = 1.0538**로 나타났으며, 이는 **Y**에 유의하지 않은 요인임을 의미한다. (**z = 0.19**, **p = .853**) 즉, **국가_미국_x_가격**가 1 증가할 때 Y가 1(사건 발생)이 될 오즈는 평균적으로 약 **5.4% 증가**하는 것으로 해석된다.
- **국가_미국_x_브랜드충성도**의 회귀계수는 **B = 0.7576**, 오즈비는 **OR = 2.1331**로 나타났으며, 이는 **Y**에 유의한 요인임을 의미한다. (**z = 2.78**, **p = .005**) 즉, **국가_미국_x_브랜드충성도**가 1 증가할 때 Y가 1(사건 발생)이 될 오즈는 평균적으로 약 **113.3% 증가**하는 것으로 해석된다.
- **국가_일본_x_보습**의 회귀계수는 **B = 0.649**, 오즈비는 **OR = 1.9136**로 나타났으며, 이는 **Y**에 유의한 요인임을 의미한다. (**z = 3.53**, **p < .001**) 즉, **국가_일본_x_보습**가 1 증가할 때 Y가 1(사건 발생)이 될 오즈는 평균적으로 약 **91.4% 증가**하는 것으로 해석된다.
- **국가_일본_x_향**의 회귀계수는 **B = 0.228**, 오즈비는 **OR = 1.256**로 나타났으며, 이는 **Y**에 유의하지 않은 요인임을 의미한다. (**z = 0.57**, **p = .565**) 즉, **국가_일본_x_향**가 1 증가할 때 Y가 1(사건 발생)이 될 오즈는 평균적으로 약 **25.6% 증가**하는 것으로 해석된다.
- **국가_일본_x_발림성**의 회귀계수는 **B = 0.0187**, 오즈비는 **OR = 1.0188**로 나타났으며, 이는 **Y**에 유의하지 않은 요인임을 의미한다. (**z = 0.11**, **p = .916**) 즉, **국가_일본_x_발림성**가 1 증가할 때 Y가 1(사건 발생)이 될 오즈는 평균적으로 약 **1.9% 증가**하는 것으로 해석된다.
- **국가_일본_x_자극**의 회귀계수는 **B = -0.3476**, 오즈비는 **OR = 0.7064**로 나타났으며, 이는 **Y**에 유의하지 않은 요인임을 의미한다. (**z = -1.91**, **p = .057**) 즉, **국가_일본_x_자극**가 1 증가할 때 Y가 1(사건 발생)이 될 오즈는 평균적으로 약 **29.4% 감소**하는 것으로 해석된다.
- **국가_일본_x_용기**의 회귀계수는 **B = 0.0099**, 오즈비는 **OR = 1.01**로 나타났으며, 이는 **Y**에 유의하지 않은 요인임을 의미한다. (**z = 0.04**, **p = .965**) 즉, **국가_일본_x_용기**가 1 증가할 때 Y가 1(사건 발생)이 될 오즈는 평균적으로 약 **1.0% 증가**하는 것으로 해석된다.
- **국가_일본_x_배송**의 회귀계수는 **B = -0.276**, 오즈비는 **OR = 0.7588**로 나타났으며, 이는 **Y**에 유의하지 않은 요인임을 의미한다. (**z = -0.79**, **p = .429**) 즉, **국가_일본_x_배송**가 1 증가할 때 Y가 1(사건 발생)이 될 오즈는 평균적으로 약 **24.1% 감소**하는 것으로 해석된다.
- **국가_일본_x_기능**의 회귀계수는 **B = -0.6119**, 오즈비는 **OR = 0.5423**로 나타났으며, 이는 **Y**에 유의한 요인임을 의미한다. (**z = -3.19**, **p = .001**) 즉, **국가_일본_x_기능**가 1 증가할 때 Y가 1(사건 발생)이 될 오즈는 평균적으로 약 **45.8% 감소**하는 것으로 해석된다.
- **국가_일본_x_가격**의 회귀계수는 **B = 0.372**, 오즈비는 **OR = 1.4506**로 나타났으며, 이는 **Y**에 유의하지 않은 요인임을 의미한다. (**z = 1.21**, **p = .228**) 즉, **국가_일본_x_가격**가 1 증가할 때 Y가 1(사건 발생)이 될 오즈는 평균적으로 약 **45.1% 증가**하는 것으로 해석된다.
- **국가_일본_x_브랜드충성도**의 회귀계수는 **B = 1.0954**, 오즈비는 **OR = 2.9903**로 나타났으며, 이는 **Y**에 유의한 요인임을 의미한다. (**z = 3.88**, **p < .001**) 즉, **국가_일본_x_브랜드충성도**가 1 증가할 때 Y가 1(사건 발생)이 될 오즈는 평균적으로 약 **199.0% 증가**하는 것으로 해석된다.

> ※ 오즈비(OR)가 1보다 크면 사건 발생 오즈가 증가, 1보다 작으면 감소함을 뜻한다. 유의확률이 유의수준보다 큰(=유의하지 않은) 변수는 효과가 통계적으로 확인되지 않았으므로 오즈비 해석에 주의한다. 더미변수의 오즈비는 '기준(drop_first로 제외된) 범주' 대비 값이다.

두 모델의 설명력(R²)을 비교한 결과, 국가×속성 상호작용항을 포함한 모델(R²=0.197)이 포함하지 않은 모델(R²=0.175)보다 근소하게 높게 나타났다.    
다만 개별 속성 변수 단위로 보면 보습과 같이 두 모델 간 유의성 판정이 엇갈리는 항목이 존재한다.

국가×속성 상호작용항은 한국을 기준(reference)으로 설정되어 있으므로, 이를 단순히 제거할 경우 한국 리뷰에서 나타나는 속성별 효과가 제대로 반영되지 않을 수 있다.    
설명력이 더 높고 이러한 위험을 피할 수 있다는 점에서, 국가×속성 상호작용항을 포함한 모델을 최종 모델로 채택하였다.

또한 개별 독립변수의 유의성이 국가×속성 상호작용항의 포함 여부에 따라 달라지므로, 로지스틱 회귀 결과만으로 변수의 채택·제거 여부를 일괄적으로 판단하기는 어렵다고 판단하였다.    
이에 따라 기존 국가×속성 상호작용항을 제거하지 않고 전체 모델에 그대로 투입하기로 결정하였다.

## 제품 통제 후 유지 여부 - 모형 C
- 모형 C는 모형 B(국가+9개 속성+국가×속성 상호작용)에서 국가 주효과 더미를 제거하고, 대신 국가와 제품_순위를 결합한 제품_ID 고정효과(더미)를 추가해 제품 자체의 고유한 만족도 차이(제품 교란)를 통제한 모형이다.

- 국가 주효과를 제거한 이유는 제품_ID가 국가 정보를 완전히 포함하고 있어 국가 더미와 완전공선성이 발생하기 때문이며, 속성 주효과와 국가×속성 상호작용항은 리뷰별로 속성 언급 여부가 달라 제품_ID와 겹치지 않으므로 그대로 유지하였다.

In [18]:
# ===== 모형 C: 모형 B + 제품 통제(제품_ID 고정효과) =====

# 1) 제품_ID 생성 (국가 + 제품_순위 결합, 머신러닝 섹션과 동일 정의)
df_final['제품_ID'] = df_final['국가'].astype(str) + '_' + df_final['제품_순위'].astype(str)

# 2) 제품_ID 더미화 (기준 제품 1개는 drop)
product_dummies = pd.get_dummies(df_final['제품_ID'], prefix='제품', drop_first=True).astype(int)

# 3) 모형 C 설계행렬: 속성 9개 + 국가×속성 27개(국가 주효과는 제외) + 제품_ID 더미
X_modelC = pd.concat(
    [df_final[attrs + interaction_cols], product_dummies, df_final['Y']],
    axis=1
)

print('모형 C 설계행렬 shape:', X_modelC.shape)

# 4) 적합 — X_modelC에 Y를 그대로 둔 채로 넘겨야 함 (fit_model이 내부에서 분리함)
fit_modelC = my_logit.fit_model(X_modelC, y='Y')

모형 C 설계행렬 shape: (6756, 76)


## 사전 가설 4개 최종 판정표 (모형 B vs 모형 C)

In [19]:
# =========================================================
# 사전 가설 4개 최종 판정표
# =========================================================
# 판정 기준:
# - 사전에 예상한 방향이 "긍정 효과"이므로 OR > 1
# - 동시에 p < .05
# 를 충족하면 '지지', 아니면 '미지지'
#
# 모형 B 판정과 제품 통제 모형 C 판정을 함께 제시한다.
# =========================================================

def country_or_from_fit(
    fit,
    attr,
    country,
    base_country="한국",
    alpha=0.05,
):
    # 기준국가(한국)는 속성 계수만, 나머지 국가는 속성 + 국가×속성 계수 합으로 검정
    if country == base_country:
        combo = attr
    else:
        combo = f"{attr} + 국가_{country}_x_{attr}"

    tt = fit.t_test(combo)
    coef = np.atleast_1d(tt.effect)[0]
    pval = np.atleast_1d(tt.pvalue)[0]
    ci_low, ci_high = tt.conf_int(alpha=alpha)[0]

    return {
        "OR": np.exp(coef),
        "p값": pval,
        "OR 95% CI 하한": np.exp(ci_low),
        "OR 95% CI 상한": np.exp(ci_high),
    }


hypotheses_final = [
    {
        "가설": "가설 1",
        "사전 예측": "향 → 일본",
        "속성": "향",
        "국가": "일본",
    },
    {
        "가설": "가설 2",
        "사전 예측": "기능 → 미국",
        "속성": "기능",
        "국가": "미국",
    },
    {
        "가설": "가설 3",
        "사전 예측": "보습 → 러시아",
        "속성": "보습",
        "국가": "러시아",
    },
    {
        "가설": "가설 4",
        "사전 예측": "가격 → 한국",
        "속성": "가격",
        "국가": "한국",
    },
]

hypothesis_rows = []

for h in hypotheses_final:

    result_b = country_or_from_fit(
        fit2,
        attr=h["속성"],
        country=h["국가"],
    )

    result_c = country_or_from_fit(
        fit_modelC,
        attr=h["속성"],
        country=h["국가"],
    )

    supported_b = (
        result_b["OR"] > 1
        and result_b["p값"] < 0.05
    )

    supported_c = (
        result_c["OR"] > 1
        and result_c["p값"] < 0.05
    )

    decision_b = (
        "지지"
        if supported_b
        else "미지지"
    )

    decision_c = (
        "지지"
        if supported_c
        else "미지지"
    )

    hypothesis_rows.append({
        "가설": h["가설"],
        "사전 예측": h["사전 예측"],
        "모형 B OR (95% CI)": (
            f'{result_b["OR"]:.2f}'
            f'{"*" if result_b["p값"] < 0.05 else " n.s."}'
            f' ({result_b["OR 95% CI 하한"]:.2f}'
            f'-{result_b["OR 95% CI 상한"]:.2f})'
        ),
        "p_모형 B": result_b["p값"],
        "모형 B 판정": decision_b,
        "제품 통제 모형 C OR (95% CI)": (
            f'{result_c["OR"]:.2f}'
            f'{"*" if result_c["p값"] < 0.05 else " n.s."}'
            f' ({result_c["OR 95% CI 하한"]:.2f}'
            f'-{result_c["OR 95% CI 상한"]:.2f})'
        ),
        "p_모형 C": result_c["p값"],
        "제품 통제 후 판정": decision_c,
        "판정 유지 여부": (
            "유지"
            if decision_b == decision_c
            else "변경"
        ),
    })

hypothesis_table = pd.DataFrame(
    hypothesis_rows
)

display(hypothesis_table)

print(
    "\n※ 지지 기준: 사전 가설의 예상 방향(OR > 1)과 p < .05를 동시에 충족\n"
    "※ '판정 유지'는 제품 고정효과를 추가한 뒤에도 지지/미지지 결론이 동일하다는 뜻입니다.\n"
    "※ 가설 3(보습→러시아)이 지지되더라도 다른 국가의 보습 OR이 더 클 수 있으므로, "
    "'러시아만의 고유 특성'으로 해석하지 않습니다."
)

,가설,사전 예측,모형 B OR (95% CI),p_모형 B,모형 B 판정,제품 통제 모형 C OR (95% CI),p_모형 C,제품 통제 후 판정,판정 유지 여부
0,가설 1,향 → 일본,1.32 n.s. (0.83-2.11),2.419803e-01,미지지,0.73 n.s. (0.42-1.26),2.567772e-01,미지지,유지
1,가설 2,기능 → 미국,2.85* (2.28-3.55),1.410143e-20,지지,3.14* (2.50-3.95),1.737469e-22,지지,유지
2,가설 3,보습 → 러시아,2.10* (1.46-3.04),7.497472e-05,지지,2.21* (1.48-3.28),8.910802e-05,지지,유지
3,가설 4,가격 → 한국,1.26 n.s. (0.88-1.81),2.132930e-01,미지지,1.28 n.s. (0.89-1.84),1.894375e-01,미지지,유지



※ 지지 기준: 사전 가설의 예상 방향(OR > 1)과 p < .05를 동시에 충족
※ '판정 유지'는 제품 고정효과를 추가한 뒤에도 지지/미지지 결론이 동일하다는 뜻입니다.
※ 가설 3(보습→러시아)이 지지되더라도 다른 국가의 보습 OR이 더 클 수 있으므로, '러시아만의 고유 특성'으로 해석하지 않습니다.


## 머신러닝용 데이터셋 저장

In [20]:
df_final.to_excel('ml_df.xlsx', index=False)
